# ReLive Knowledge Base Test Harness

This notebook tests the full write/read workflow:

1. Load sample texts from `data/`
2. Write extracted knowledge into Neo4j using the writer graph
3. Run chat-like read queries against the knowledge base

## Multi-Concept Extraction

The writer now extracts **multiple distinct concepts** from each text. If a text contains multiple:
- Environments
- Problems  
- Solutions
- Results

...each combination is extracted as a separate knowledge entry. For example, a text describing 3 different solutions creates 3 separate knowledge entries in the database.

## Provider Configuration

- **LLM**: Choose from `"openai"`, `"gemini"`, `"inception"` (Mercury 2), or `"none"` for deterministic local mode
- **Embedder**: Choose from `"openai"`, `"gemini"`, or `"qwen"` (local fallback)
  - **Qwen3 Embedding** automatically downloads on first use (no API key needed)
  - Falls back from cloud providers if not configured

## Neo4j Configuration

- **Target**: Set `NEO4J_TARGET` in the setup cell to `"local"`, `"hosted"`, or `"docker"`
- **Hosted (Aura)**: Add to `.env`:
  - `NEO4J_HOSTED_NAME` — instance id (e.g. `97f8db06`)
  - `NEO4J_HOSTED_PASSWORD` — database password
  - Optional: `NEO4J_HOSTED_URI`, `NEO4J_HOSTED_USERNAME`, `NEO4J_HOSTED_DATABASE`

## Quick Fix: Neo4j Aura Authentication

If you get `AuthError: The client is unauthorized due to authentication failure`:

1. **Reset Aura Password**:
   - Go to: https://console.neo4j.io
   - Click your instance (e.g., **HIVE**)
   - Go to: Details → Security → Reset password
   - Copy the new password

2. **Update `.env`**:
   ```env
   NEO4J_TARGET=hosted
   NEO4J_HOSTED_NAME=7477e481  # Your instance ID
   NEO4J_HOSTED_PASSWORD=<NEW_PASSWORD>
   ```

3. **Restart**:
   ```bash
   docker-compose down
   docker-compose up -d
   ```

4. **Restart Jupyter kernel**: Kernel → Restart


In [3]:
from __future__ import annotations

import json
from pathlib import Path
from math import floor

from src.logic.orchestrator import AgentOrchestrator
from src.ai.providers import (
    GeminiEmbedderClient,
    GeminiLLMClient,
    GeminiProviderConfig,
    OpenAIEmbedderClient,
    OpenAILLMClient,
    OpenAIProviderConfig,
    InceptionEmbedderClient,
    InceptionLLMClient,
    InceptionProviderConfig,
    QwenEmbedderClient,
    QwenProviderConfig,
)
from src.ai.embedder_utils import get_embedder_safe
from src.ai.chunk_utils import TextSplitter, word_counter

In [4]:
# Choose provider: "none", "openai", "gemini", "inception" (LLM), or "qwen" (embeddings only).
PROVIDER = "inception"
EMBEDDER_PROVIDER = "qwen"  # Use "qwen" for local embeddings, or "openai"/"gemini"

# Neo4j: "local" (Docker/localhost), "hosted" (Aura / remote), or in-memory debug.
NEO4J_TARGET = "hosted"  # "local" | "hosted" | "docker"
USE_IN_MEMORY_DEBUG = False

import importlib
import os
from pathlib import Path

from dotenv import load_dotenv

# Load .env from project root (Jupyter cwd is often not the repo root)
_PROJECT_ROOT = Path.cwd()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "pyproject.toml").exists() or (candidate / "src" / "database").is_dir():
        _PROJECT_ROOT = candidate
        break
load_dotenv(_PROJECT_ROOT / ".env", override=False)

if USE_IN_MEMORY_DEBUG:
    os.environ["NEO4J_DEBUG_IN_MEMORY"] = "true"
else:
    os.environ["NEO4J_TARGET"] = NEO4J_TARGET

# Reload modules in case notebook kernel cached older code.
import src.database.config as db_config
import src.database.infrastructure.driver as neo_driver
import src.database.manager as db_manager
import src.logic.orchestrator as orchestrator_mod
importlib.reload(db_config)
importlib.reload(neo_driver)
importlib.reload(db_manager)
importlib.reload(orchestrator_mod)
neo_driver.Neo4jDriver._instance = None

llm = None
embedder = None

# Setup LLM provider
if PROVIDER == "openai":
    cfg = OpenAIProviderConfig.from_env()
    llm = OpenAILLMClient(cfg)
elif PROVIDER == "gemini":
    cfg = GeminiProviderConfig.from_env()
    llm = GeminiLLMClient(cfg)
elif PROVIDER == "inception":
    cfg = InceptionProviderConfig()
    llm = InceptionLLMClient(cfg)

# Setup Embedder provider (with fallback to Qwen if not configured)
if EMBEDDER_PROVIDER == "openai":
    if llm is None or not isinstance(llm, OpenAILLMClient):
        cfg = OpenAIProviderConfig.from_env()
    embedder = OpenAIEmbedderClient(cfg)
elif EMBEDDER_PROVIDER == "gemini":
    if llm is None or not isinstance(llm, GeminiLLMClient):
        cfg = GeminiProviderConfig.from_env()
    embedder = GeminiEmbedderClient(cfg)
elif EMBEDDER_PROVIDER == "qwen":
    cfg = QwenProviderConfig()
    embedder = QwenEmbedderClient(cfg)
else:
    # Default: use safe fallback (tries None first, then Qwen)
    embedder = get_embedder_safe(primary_embedder=None, fallback_to_qwen=True)

from src.database.config import Neo4jSettings, load_project_env
from src.database.manager import DatabaseManager
load_project_env(override=False)
neo4j_settings = Neo4jSettings()
if neo4j_settings.target == "hosted" and not neo4j_settings.credentials_ok_for_hosted():
    raise RuntimeError(
        "Hosted Neo4j password not loaded. Check .env has NEO4J_HOSTED_PASSWORD "
        f"and restart the kernel. Project root: {_PROJECT_ROOT}"
    )
orchestrator = AgentOrchestrator(
    llm=llm,
    embedder=embedder,
    db=DatabaseManager(settings=neo4j_settings),
    top_k=10
)
await orchestrator.initialize()
print("Orchestrator initialized")
print(f"  LLM Provider: {PROVIDER}")
print(f"  Embedder Provider: {EMBEDDER_PROVIDER}")
print(f"  Neo4j: {neo4j_settings.connection_summary()}")
print(f"  Debug Mode: {USE_IN_MEMORY_DEBUG}")

Orchestrator initialized
  LLM Provider: inception
  Embedder Provider: qwen
  Neo4j: hosted (neo4j+s://…98170365.databases.neo4j.io, db=98170365, user=98170365, creds=ok)
  Debug Mode: False


In [5]:
# Diagnostic: Test LLM and Embedder provider connectivity
print("🔍 Testing Providers...\n")

# Test LLM
print("LLM Provider:")
if llm is None:
    print("  ⚠️  No LLM provider configured")
    print("     The system will use heuristic fallbacks for extraction/reflection")
else:
    try:
        test_prompt = "Respond with exactly: {\"test\": true}"
        test_response = await llm.ainvoke(test_prompt)
        print(f"  ✓ LLM responding ({len(test_response)} chars)")
        if not test_response.strip():
            print("  ⚠️  WARNING: LLM returned empty response!")
    except Exception as e:
        print(f"  ✗ LLM error: {e}")

# Test Embedder
print("\nEmbedding Provider:")
if embedder is None:
    print("  ✗ No embedder available!")
else:
    try:
        test_embedding = await embedder.embed("test")
        print(f"  ✓ Embedder responding ({len(test_embedding)}-dim)")
    except Exception as e:
        print(f"  ✗ Embedder error: {e}")

🔍 Testing Providers...

LLM Provider:
  ✓ LLM responding (14 chars)

Embedding Provider:
Loading Qwen3 Embedding model from: /home/r2/Documents/Projects/HIVE/backend/models/qwen
Model: Qwen/Qwen3-Embedding-0.6B
Device: cuda


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

✓ Embedding model loaded successfully on cuda
  ✓ Embedder responding (1024-dim)


In [6]:
data_dir = Path("data")
files = sorted(data_dir.glob("*.txt"))
print("Sample files:")
for f in files:
    print("-", f)

texts = [{"name": f.name, "text": f.read_text(encoding="utf-8")} for f in files]
print(f"Loaded {len(texts)} text documents")

Sample files:
Loaded 0 text documents


In [18]:
async def chat_turn(user_message: str, top_k: int | None = None) -> dict:
    """Run a single chat-like retrieval turn against the knowledge base."""
    response = await orchestrator.run(mode="retrieve", text=user_message, top_k=top_k)
    return response.get("response", response)


def pretty_print_turn(query: str, payload: dict) -> None:
    print(f"USER: {query}\n")
    # parse payload from json string to dictionary
    payload_object = payload if isinstance(payload, dict) else json.loads(payload)
    ranked = payload_object.get("ranked_results", [])
    if not ranked:
        print("ASSISTANT: No close matches found.")
        return

    best = ranked[0]
    print("ASSISTANT: Top match")
    print(json.dumps(best, indent=2))
    print("\nALTERNATIVES:")
    print(json.dumps(payload_object.get("alternatives", []), indent=2))

print(
    "What ilnesses can cause fever?",
    await chat_turn("What ilnesses can cause fever?", top_k=10)
)

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `ALTERNATIVE_SOLUTION` does not exist in database `98170365`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=21, offset=21>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 21, 'line': 2, 'column': 21}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        MATCH (n)-[:ALTERNATIVE_SOLUTION|ALTERNATIVE_RESULT|LEADS_TO_RESULT|WORKS_IN_ENVIRONMENT*1..2]-(alt)\n        WHERE elementId(n) = $node_id\n        RETURN elementId(alt) AS alternative_id,\n               labels(alt) AS labels,\n               coalesce(alt.text, '') AS text

What ilnesses can cause fever? {
  "response": "Fever is a common sign of many illnesses, especially those that involve infection or inflammation. Based on the available evidence, notable causes include: (1) acute infections of any type (bacterial, viral, fungal); (2) recurrent or chronic infections that persist or recur; (3) Dressler’s syndrome, an inflammatory reaction after a heart attack that can produce fever; (4) complications of diverticular disease that lead to infection or hemorrhage; (5) painful perianal infections that can spread; and (6) severe malnutrition, which lowers immunity and predisposes to infectious fever.",
  "confidence": "low",
  "confidence_reasoning": "The ranked concepts mainly describe generic problem statements (e.g., \"potential infection\", \"recurrent infection\", \"Dressler's syndrome\") rather than a comprehensive, high‑grade list of specific illnesses. The evidence is relevant but limited and not strongly graded, so the answer is synthesized with cau

In [17]:
print(
    "What are symptoms of Dressler's syndrome?", 
    await chat_turn(
        user_message="What are symptoms of Dressler's syndrome?",
        top_k=10
    )
)

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `ALTERNATIVE_SOLUTION` does not exist in database `98170365`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=21, offset=21>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 21, 'line': 2, 'column': 21}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        MATCH (n)-[:ALTERNATIVE_SOLUTION|ALTERNATIVE_RESULT|LEADS_TO_RESULT|WORKS_IN_ENVIRONMENT*1..2]-(alt)\n        WHERE elementId(n) = $node_id\n        RETURN elementId(alt) AS alternative_id,\n               labels(alt) AS labels,\n               coalesce(alt.text, '') AS text

What are symptoms of Dressler's syndrome? {
  "response": "Dressler’s syndrome typically presents weeks after a myocardial infarction with recurrent fever, chest pain, and a pericardial friction rub; it can also progress to cardiac tamponade.",
  "confidence": "high",
  "confidence_reasoning": "The top‑ranked concept directly lists the core clinical features of Dressler’s syndrome, providing a clear and specific symptom set.",
  "applied_concepts": ["Dressler's syndrome symptom triad"],
  "caveats": "Symptoms appear weeks post‑MI and may vary; cardiac tamponade is a potential complication, not a universal symptom."
}


In [ ]:
ingest_results = []
for item in texts:
    try:
        result = await orchestrator.run(mode="write", text=item["text"], environment_hint="auto")
    except Exception:
        number_of_words = word_counter(item["text"])
        splitter = TextSplitter(
            chunk_size=int(floor(number_of_words * 0.5)),
            chunk_overlap=int(floor(number_of_words * 0.025))
        )
        chunks = splitter.produce_chunks(item["text"])
        for chunk in chunks:
            result = await orchestrator.run(mode="write", text=chunk, environment_hint="auto")
            persisted_list = result.get("persisted", [])
            ingest_results.append({
                "file": item["name"],
                "num_concepts_extracted": len(persisted_list),
                "knowledge_entry_ids": [entry.get("knowledge_entry_id") for entry in persisted_list],
                "errors": result.get("errors", []),
            })

print(json.dumps(ingest_results, indent=2))